<a href="https://colab.research.google.com/github/VickkiMars/papers/blob/main/FreEformer_BTC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
import torch.nn as nn
import numpy as np

In [ ]:
import os

os.environ['KAGGLE_USERNAME'] = "victorthamartian"
os.environ['KAGGLE_KEY'] = "928e769b1a2543bce53ee9709c015582"


In [ ]:
!kaggle datasets download -d mczielinski/bitcoin-historical-data

Dataset URL: https://www.kaggle.com/datasets/mczielinski/bitcoin-historical-data
License(s): CC-BY-SA-4.0
  0% 0.00/97.7M [00:00<?, ?B/s]
100% 97.7M/97.7M [00:00<00:00, 1.49GB/s]


In [ ]:
!unzip /content/bitcoin-historical-data.zip

Archive:  /content/bitcoin-historical-data.zip
  inflating: btcusd_1-min_data.csv   


In [4]:
class LogLayer(nn.Module):
  def forward(self, x):
    x = torch.clamp(x, min=1e-5)
    x = torch.log(x)
    return x

In [ ]:
import pandas as pd
btc = pd.read_csv("/content/btcusd_1-min_data.csv")

In [ ]:
btc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7281757 entries, 0 to 7281756
Data columns (total 6 columns):
 #   Column     Dtype  
---  ------     -----  
 0   Timestamp  float64
 1   Open       float64
 2   High       float64
 3   Low        float64
 4   Close      float64
 5   Volume     float64
dtypes: float64(6)
memory usage: 333.3 MB


In [ ]:
import torch.nn.functional as F
import math

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class EnhancedAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.scale = math.sqrt(d_model)
        self.learnable_matrix = nn.Parameter(torch.randn(d_model, d_model))
        self.softplus = nn.Softplus()

    def forward(self, Q, K, V):
        # Q, K, V: [batch, time, d_model]
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        attn_weights = F.softmax(attn_scores, dim=-1)

        # Weighted sum in value space -> [B, T, D]
        out = torch.matmul(attn_weights, V)

        # Apply learnable transformation in feature space
        mod_matrix = self.softplus(self.learnable_matrix)
        out = torch.matmul(out, mod_matrix)

        # Row-wise L1 normalization
        out = out / (out.abs().sum(dim=-1, keepdim=True) + 1e-8)
        return out


In [6]:
class EnhancedTransformerBlock(nn.Module):
  def __init__(self, d_model, d_ff=256):
    super().__init__()
    self.attn = EnhancedAttention(d_model)
    self.norm1 = nn.LayerNorm(d_model)
    self.ffn = nn.Sequential(
        nn.Linear(d_model, d_ff),
        nn.GELU(),
        nn.Linear(d_ff, d_model)
    )
    self.norm2 = nn.LayerNorm(d_model)

  def forward(self, x):
    attn_out = self.attn(x, x, x)
    x = self.norm1(x + attn_out)

    # FFN + Residual
    ffn_out = self.ffn(x)
    x = self.norm2(x + ffn_out)
    return x

In [7]:
class FreEformer(nn.Module):
  def __init__(self, input_dim, d_model, output_dim, forecast_steps, num_blocks=2):
    super().__init__()
    self.input_norm = nn.InstanceNorm1d(input_dim)
    self.embed = nn.Linear(input_dim, d_model)

    self.transformer_blocks = nn.ModuleList([
        EnhancedTransformerBlock(d_model) for _ in range(num_blocks)
    ])

    self.proj_out = nn.Linear(d_model, output_dim)
    self.inverse_norm = nn.InstanceNorm1d(output_dim)
    self.forecast_steps = forecast_steps

  def forward(self, x):
    """
    x: [batch, seq_len, input_dim]
    """

    x = x.permute(0, 2, 1)
    x = self.input_norm(x)
    x = x.permute(0, 2, 1)

    # Dimension Extension
    x_emb = self.embed(x)
    x_skip_time = x_emb.clone()

    # DFT (along time axis)
    x_freq = torch.fft.fft(x_emb, dim=1)

    # Enhanced Transformer Blocks
    for block in self.transformer_blocks:
      x_freq = block(x_freq.real) + 1j * block(x_freq.imag) # process real and imag parts

    # iDFT and skip connection
    x_time = torch.fft.ifft(x_freq, dim=1).real
    x_time = x_time + x_skip_time

    # forecast next k steps
    # simple extrapolation via a linear projection of the final hidden state

    last_state = x_time[:, -1:, :]
    forecasts = []

    for _ in range(self.forecast_steps):
      next_step = self.proj_out(last_state)
      forecasts.append(next_step)
      last_state = self.embed(next_step) # autoregressive rollout
    #forecasts = self.proj_out(last_state.repeat(1, self.forecast_steps, 1))

    y_pred = torch.cat(forecasts, dim=1)

    # inverse InstanceNorm
    y_pred = y_pred.permute(0, 2, 1)
    y_pred = self.inverse_norm(y_pred)
    y_pred = y_pred.permute(0, 2, 1)

    return y_pred

## Freeformer with LogLayer

In [27]:
class LogLayer(nn.Module):
    def forward(self, x):
        return torch.log(torch.clamp(x, min=1e-6))


class FreEformer(nn.Module):
    def __init__(self, input_dim, d_model, output_dim, forecast_steps, num_blocks=2):
        super().__init__()
        self.input_norm = nn.InstanceNorm1d(input_dim)
        self.embed = nn.Linear(input_dim, d_model)

        self.log_layer = LogLayer()

        self.transformer_blocks = nn.ModuleList([
            EnhancedTransformerBlock(d_model) for _ in range(num_blocks)
        ])

        self.proj_out = nn.Linear(d_model, output_dim)
        self.inverse_norm = nn.InstanceNorm1d(output_dim)
        self.forecast_steps = forecast_steps

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.input_norm(x)
        x = x.permute(0, 2, 1)

        x_emb = self.embed(x)

        # ---- FFT WITH LOG-MAGNITUDE ----
        x_freq = torch.fft.fft(x_emb, dim=1)
        mag = torch.abs(x_freq)
        phase = torch.angle(x_freq)

        mag = self.log_layer(mag)

        x_freq = mag * torch.exp(1j * phase)

        # ---- Transformer blocks on real/imag ----
        for block in self.transformer_blocks:
            real = block(x_freq.real)
            imag = block(x_freq.imag)
            x_freq = real + 1j * imag

        # ---- iFFT ----
        x_time = torch.fft.ifft(x_freq, dim=1).real
        x_time = x_time + x_emb  # skip

        # ---- Forecasting ----
        last_state = x_time[:, -1:, :]
        forecasts = []
        for _ in range(self.forecast_steps):
            next_step = self.proj_out(last_state)
            forecasts.append(next_step)
            last_state = self.embed(next_step)

        y_pred = torch.cat(forecasts, dim=1)

        y_pred = y_pred.permute(0, 2, 1)
        y_pred = self.inverse_norm(y_pred)
        y_pred = y_pred.permute(0, 2, 1)

        return y_pred


In [28]:
batch_size = 32
input_len = 512
input_dim = 5
output_dim = 5
forecast_len = 64
d_model = 128
epochs = 10
lr = 1e-4
device = "cuda" if torch.cuda.is_available() else "cpu"

In [29]:
import torch
from torch.utils.data import Dataset, DataLoader

In [30]:
class BitcoinDataset(Dataset):
  def __init__(self, df, input_len=512, forecast_len=64, feature_cols=None):
    if feature_cols is None:
      feature_cols = ['Open', 'High', 'Low', 'Close', 'Volume']

    self.data = df.sort_values('Timestamp')[feature_cols].values
    self.input_len = input_len
    self.forecast_len = forecast_len

    self.mean = self.data.mean(axis=0, keepdims=True)
    self.std = self.data.std(axis=0, keepdims=True)
    self.data = (self.data - self.mean) / (self.std + 1e-8)

  def __len__(self):
    return len(self.data) - self.input_len - self.forecast_len

  def __getitem__(self, idx):
    x = self.data[idx : idx + self.input_len]
    y = self.data[idx + self.input_len : idx + self.input_len + self.forecast_len]
    return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

In [31]:
import numpy as np
import pandas as pd

# parameters
input_len = 512
forecast_len = 64
dataset_len = 1_000_000
num_rows = dataset_len + input_len + forecast_len   # 1,000,576

# ---- Synthetic data generation ----
np.random.seed(42)

# Generate a synthetic price series with random walk + volatility
price = np.cumsum(np.random.normal(loc=0.0, scale=1.0, size=num_rows)) + 30000

# High/Low around price
high = price + np.random.uniform(0.1, 5.0, size=num_rows)
low  = price - np.random.uniform(0.1, 5.0, size=num_rows)

# Open/Close slightly noisy versions of price
open_price  = price + np.random.normal(0, 1.0, size=num_rows)
close_price = price + np.random.normal(0, 1.0, size=num_rows)

# Volume: positive, log-normal distribution
volume = np.random.lognormal(mean=12, sigma=1.0, size=num_rows)

# Timestamps: 1-min increments (or any frequency)
timestamps = np.arange(num_rows)

# ---- Create DataFrame ----
df = pd.DataFrame({
    "Timestamp": timestamps,
    "Open": open_price,
    "High": high,
    "Low": low,
    "Close": close_price,
    "Volume": volume,
})

print(df.head())
print("Rows:", len(df))


   Timestamp          Open          High           Low         Close  \
0          0  30001.002202  30000.694709  29998.167809  30000.618487   
1          1  29998.625529  30004.808656  29997.080334  29999.636035   
2          2  30001.473493  30002.888333  29996.990308  30000.178323   
3          3  30002.994778  30003.579821  30002.311726  30002.131216   
4          4  30003.578243  30006.567456  29999.527226  30002.849934   

          Volume  
0  150998.268380  
1  102419.738677  
2  217421.022269  
3  215290.365824  
4  139694.512737  
Rows: 1000576


In [32]:
full_dataset = BitcoinDataset(df, input_len=512, forecast_len=64)

#loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)
VAL_SPLIT=0.1
BATCH_SIZE=32
from torch.utils.data import random_split
val_size = int(len(full_dataset) * VAL_SPLIT)
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [33]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class BTCWindowDataset(Dataset):
    def __init__(self, data, seq_len=512, forecast_len=64, stride=32):
        self.seq_len = seq_len
        self.forecast_len = forecast_len
        self.stride = stride

        # Save mean/std
        self.mean = data.mean(axis=0, keepdims=True)
        self.std = data.std(axis=0, keepdims=True)

        # Normalize
        data = (data - self.mean) / (self.std + 1e-8)

        self.data = torch.tensor(data, dtype=torch.float32)

        self.indices = np.arange(0, len(data) - seq_len - forecast_len, stride)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        x = self.data[i:i+self.seq_len]
        y = self.data[i+self.seq_len:i+self.seq_len+self.forecast_len]
        return x, y


In [34]:
# assuming df is your full BTC dataframe
data = df[["Open", "High", "Low", "Close", "Volume"]].values

train_ratio = 0.9
split_idx = int(len(data) * train_ratio)

train_data = data[:split_idx]
val_data = data[split_idx:]

seq_len = 512
forecast_len = 64
stride = 16  # <-- reduces dataset size by ~32×

train_dataset = BTCWindowDataset(train_data, seq_len, forecast_len, stride)
val_dataset = BTCWindowDataset(val_data, seq_len, forecast_len, stride)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

print(f"✅ Train samples: {len(train_dataset):,}")
print(f"✅ Val samples: {len(val_dataset):,}")


✅ Train samples: 56,247
✅ Val samples: 6,218


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [35]:
model = FreEformer(
    input_dim=input_dim,
    d_model=d_model,
    output_dim=output_dim,
    forecast_steps=forecast_len,
    num_blocks=4
).to(device)

In [36]:
import torch.optim as optim
from tqdm import tqdm
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

In [37]:
from tqdm import tqdm
import time
import torch
import os

# === CONFIG ===
EPOCHS, DEVICE = epochs, device
CHECKPOINT_INTERVAL = 15 * 60  # 15 minutes
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

start_time = time.time()
last_checkpoint_time = start_time

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    # tqdm progress bar per epoch
    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{EPOCHS}", ncols=120)

    for batch_idx, (x, y) in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        y_pred = model(x)
        loss = criterion(y_pred, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Update progress bar
        avg_loss = running_loss / (batch_idx + 1)
        pbar.set_postfix({"avg_loss": f"{avg_loss:.6f}"})

        # Checkpoint every 15 minutes
        current_time = time.time()
        if current_time - last_checkpoint_time >= CHECKPOINT_INTERVAL:
            elapsed_mins = int((current_time - start_time) / 60)
            ckpt_path = os.path.join(
                CHECKPOINT_DIR,
                f"freeformer_epoch{epoch+1}_iter{batch_idx+1}_{elapsed_mins}min.pt"
            )
            torch.save({
                "epoch": epoch,
                "batch": batch_idx,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "loss": avg_loss,
                "elapsed_minutes": elapsed_mins,
            }, ckpt_path)
            tqdm.write(f" Saved checkpoint: {ckpt_path}")
            last_checkpoint_time = current_time

    # === End of epoch ===
    avg_loss = running_loss / len(train_loader)
    tqdm.write(f"Epoch [{epoch+1}/{EPOCHS}] - Avg Loss: {avg_loss:.6f}")

    # Save final checkpoint for epoch
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": avg_loss,
    }, os.path.join(CHECKPOINT_DIR, f"freeformer_epoch{epoch+1}_final.pt"))

    tqdm.write(f"Saved final checkpoint for epoch {epoch+1}")


Epoch 1/10: 100%|████████████████████████████████████████████████| 1758/1758 [03:17<00:00,  8.89it/s, avg_loss=1.105416]

Epoch [1/10] - Avg Loss: 1.105416
Saved final checkpoint for epoch 1



Epoch 2/10: 100%|████████████████████████████████████████████████| 1758/1758 [03:15<00:00,  8.98it/s, avg_loss=1.005153]

Epoch [2/10] - Avg Loss: 1.005153
Saved final checkpoint for epoch 2



Epoch 3/10: 100%|████████████████████████████████████████████████| 1758/1758 [03:16<00:00,  8.97it/s, avg_loss=1.002927]

...

 Saved checkpoint: checkpoints/freeformer_epoch5_iter1019_15min.pt


Epoch 5/10: 100%|████████████████████████████████████████████████| 1758/1758 [03:16<00:00,  8.96it/s, avg_loss=1.001746]

Epoch [5/10] - Avg Loss: 1.001746
Saved final checkpoint for epoch 5



Epoch 6/10:   5%|██▋                                               | 96/1758 [00:10<03:06,  8.90it/s, avg_loss=0.990732]


KeyboardInterrupt: 

In [ ]:
checkpoint = torch.load("checkpoints/freeformer_epoch3_iter200_45min.pt")
model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
start_epoch = checkpoint["epoch"]


In [ ]:
import os
import time
import torch

# === CONFIG ===
EPOCHS = 10
CHECKPOINT_INTERVAL = 15 * 60  # 15 minutes
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === OPTIONAL: helper to get latest checkpoint ===
def get_latest_checkpoint(ckpt_dir):
    ckpts = [os.path.join(ckpt_dir, f) for f in os.listdir(ckpt_dir) if f.endswith(".pt")]
    if not ckpts:
        return None
    ckpts.sort(key=os.path.getmtime)  # most recent last
    return ckpts[-1]

# === Attempt resume ===
start_epoch = 0
last_checkpoint_time = time.time()

latest_ckpt = get_latest_checkpoint(CHECKPOINT_DIR)
if latest_ckpt:
    print(f"🔄 Resuming from checkpoint: {latest_ckpt}")
    checkpoint = torch.load(latest_ckpt, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    last_checkpoint_time = time.time() - CHECKPOINT_INTERVAL  # force early save
else:
    print("🆕 No checkpoint found. Starting fresh training.")

# === TRAIN LOOP ===
for epoch in range(start_epoch, EPOCHS):
    model.train()
    running_loss = 0.0
    start_time = time.time()

    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        y_pred = model(x)
        loss = criterion(y_pred, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # === Checkpoint every 15 minutes ===
        current_time = time.time()
        if current_time - last_checkpoint_time >= CHECKPOINT_INTERVAL:
            elapsed_mins = int((current_time - start_time) / 60)
            ckpt_path = os.path.join(
                CHECKPOINT_DIR,
                f"freeformer_epoch{epoch+1}_iter{batch_idx+1}_{elapsed_mins}min.pt"
            )
            torch.save({
                "epoch": epoch,
                "batch": batch_idx,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "loss": running_loss / (batch_idx + 1),
                "elapsed_minutes": elapsed_mins,
            }, ckpt_path)
            print(f"✅ Saved checkpoint: {ckpt_path}")
            last_checkpoint_time = current_time

    avg_loss = running_loss / len(train_loader)
    print(f"📘 Epoch [{epoch+1}/{EPOCHS}] | Avg Loss: {avg_loss:.6f}")

    # === Final checkpoint each epoch ===
    ckpt_final = os.path.join(CHECKPOINT_DIR, f"freeformer_epoch{epoch+1}_final.pt")
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": avg_loss,
    }, ckpt_final)
    print(f"💾 Saved final checkpoint: {ckpt_final}")
